# The Genomic Bottleneck — hands-on

**SIFR-2026 · MorphoNAS**

A human genome (~700 MB) grows a brain of ~100 trillion synapses that works
*at birth*, with no training data. That compression — a compact developmental
program, not a giant weight vector — is the **genomic bottleneck**.

In the next ~75 minutes you'll do it yourself with **MorphoNAS**: grow neural
networks from compact genomes, find ones that already control CartPole with zero
learning, and evolve a tiny ~7-neuron controller under size pressure.

### How to run
- **Runtime is CPU-only — that's fine** (no GPU needed). Make sure you're **signed into Google**.
- Run the **Setup** cell first (it installs the package on the Colab VM — about 1–2 min).
- Then run cells top to bottom. Cells marked **✏️ your turn** are yours to edit.


In [ ]:
# Setup — run me first  (installs the playbook on this Colab VM)
!pip install -q "git+https://github.com/ukma-morphonas-lab/MorphoNAS-playbook.git"

import morphonas_playbook as mp
from IPython.display import Image, display
print("MorphoNAS-playbook", mp.__version__, "ready")


## 1 · Watch a network grow

A MorphoNAS genome doesn't store a network — it stores *rules*. From a single
progenitor cell, chemical morphogens diffuse across a 10×10 grid, cells divide
and differentiate into neurons, and axons grow along the gradients to wire them
up. It's **deterministic**: the same genome always grows the same network.


In [ ]:
SEED = 39      # ✏️ your turn: change the seed and re-run to grow a different genome

genome = mp.random_genome(SEED)
grid   = mp.grow(genome)
G      = mp.graph_of(grid)
print(f"seed {SEED}: {mp.n_nodes(G)} neurons, {mp.n_edges(G)} connections, "
      f"connected={mp.is_connected(G)}")

mp.growth.animate_growth(genome, "growth.gif")
display(Image("growth.gif"))


In [ ]:
# the finished wiring diagram (sources = amber, downstream = teal)
mp.growth.show_network(G, "network.png")
display(Image("network.png"))


Most random genomes grow into **junk or blobs** — either a couple of cells, or
the whole grid fills up (~100 neurons). A clean, readable mid-size network is
rare (~1–2% of random genomes); seed 39 is a curated legible one.

Want to feel that rarity? The cell below keeps growing random genomes until it
finds a legible one, and reports how many tries it took.


In [ ]:
seed, genome, grid, G, tries = mp.growth.find_connected_genome(start_seed=0)
print(f"found a legible network after {tries} tries (seed {seed})")


## 2 · Zero-learning control

Here's the surprise. Take a grown network, wire its inputs to CartPole's 4
observations and its outputs to the 2 actions — **no training, no gradient
descent** — and just run it. Some genomes already balance the pole.


In [ ]:
# a random genome that already solves CartPole — with zero learning
genome = mp.random_genome(mp.control.HERO_SOLVER_SEED)
G = mp.graph_of(mp.grow(genome))
mean, _ = mp.evaluate(G, episodes=20)
print(f"mean reward over 20 episodes: {mean:.0f} / 500   (no learning!)")

mp.control.render_rollout(G, "solver.gif")
display(Image("solver.gif"))


How common is that? Let's grow a batch of random genomes and count how many
already **beat a do-nothing policy** (reward ≥ 200 — a null policy survives only
~9 steps), among those that grow into a valid controller.


In [ ]:
stats = mp.control.measure(N=200)     # about 10–20 s


> **Optional (advanced):** is a *grown* network actually better than random
> wiring with the same neuron and edge counts? The lecture reports a **2.84×**
> size-controlled advantage. A live version of that comparison can be dropped in
> here — ask the tutor.


## 3 · Evolve a controller — the easy way

Now let's *search* for a good controller with a genetic algorithm: grow each
genome, score it on CartPole, keep the best, mutate, repeat. Because a few
percent of random genomes already work, this is almost trivial — a perfect
500-step solver usually appears in **generation 0 or 1**.


In [ ]:
result = mp.evolve.run_ga(
    penalize_connections=False,     # no size pressure (Part 3)
    population=150, seed=3, num_rollouts=10, max_workers=2,
)
print(result.summary())
# if you hit a multiprocessing error on Colab, set max_workers=1


It solved almost instantly — but look at the elite: **~100 neurons, hundreds
of connections**. With no pressure to be small, evolution has no reason to
compress. A compact genome can't store a network that big. That's the problem.


## 4 · The bottleneck bites  ✏️ your turn

Change **one thing**: turn on the size penalty. Now fitness rewards *small*
networks that still balance the pole. Watch evolution compress the ~100-neuron
monster down toward the ~6-neuron ideal — the collapse lands within a few generations
(the exact generation varies with your hardware).


In [ ]:
result = mp.evolve.run_ga(
    penalize_connections=True,      # the only change: punish big networks (Part 4)
    penalty_kwargs=mp.evolve.STRONG_PENALTY,
    population=150, seed=3, num_rollouts=10,   # this combo compresses cleanly — keep it
    max_workers=2, stop_on_solver=False,
    stop_neurons=8, max_generations=8,         # stop as soon as it's small AND still solves
)
print(result.summary())
# a couple of minutes on a Colab CPU — watch the 'neurons' column collapse 100 -> ~6.


In [ ]:
# the evolved controller: small, and it still balances
from genome import Genome
best = Genome.from_json(json_str=result.best_genome_json)
G = mp.graph_of(mp.grow(best))
print(f"evolved controller: {mp.n_nodes(G)} neurons, {mp.n_edges(G)} connections")
mp.growth.show_network(G, "evolved.png"); display(Image("evolved.png"))
mp.control.render_rollout(G, "evolved.gif"); display(Image("evolved.gif"))


### Guaranteed payoff
If your run didn't fully compress in the generations you gave it, here's one we
evolved earlier — a **6-neuron / 11-edge** controller that solves CartPole
perfectly. A few hundred bits of genome, grown into a competent controller.


In [ ]:
best = mp.assets.load_genome("part4")      # pre-evolved 6-neuron solver
G = mp.graph_of(mp.grow(best))
mean, _ = mp.evaluate(G, episodes=20)
print(f"pre-evolved: {mp.n_nodes(G)} neurons, {mp.n_edges(G)} connections, "
      f"reward {mean:.0f}/500")
mp.growth.show_network(G, "tiny.png"); display(Image("tiny.png"))
mp.control.render_rollout(G, "tiny.gif"); display(Image("tiny.gif"))


## Wrap-up

You grew networks from compact genomes, found that some **already** control
CartPole with zero learning, and evolved a **~7-neuron** controller once you
asked for *small*. The lesson of the genomic bottleneck: the compression isn't a
limitation to work around — it's *where the structure (and the generalization)
comes from*.

### Stretch ideas
- **Tune the size penalty:** replace `mp.evolve.STRONG_PENALTY` with your own
  knobs, or use `mp.evolve.SizePenaltyScorer(lam=0.05)` (penalizes neuron count
  directly): `run_ga(scorer=mp.evolve.SizePenaltyScorer(lam=0.05), ...)`.
- **Harder tasks:** the same engine grows controllers for other control tasks
  (Acrobot, MountainCar — both already installed) — they just need different
  input/output sizes, e.g. Acrobot:

  ```python
  G = mp.graph_of(mp.grow(mp.random_genome(0)))
  mean, _ = mp.evaluate(G, env_name="Acrobot-v1", episodes=20,
                        input_dim=6, output_dim=3)
  ```
